## 1. Project Information

# FinSight AI

## Notebook 03 — Master Dataset Construction

### Objective

Create one clean master dataset that will be used throughout the project.

### Input

- raw_analyst_ratings.csv
- raw_partner_headlines.csv

### Output

master_news_dataset.parquet

## 2. Imports

In [31]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## 3. Project Paths

In [32]:
PROJECT_ROOT = "/content/drive/MyDrive/FinSight_AI"

RAW_DATA = os.path.join(PROJECT_ROOT, "data/raw")

INTERIM_DATA = os.path.join(PROJECT_ROOT, "data/interim")

## 4. Load Datasets

In [33]:
analyst_df = pd.read_csv(

    os.path.join(RAW_DATA, "raw_analyst_ratings.csv"),

    low_memory=False

)

partner_df = pd.read_csv(

    os.path.join(RAW_DATA, "raw_partner_headlines.csv"),

    low_memory=False

)

print("Analyst Dataset :", analyst_df.shape)

print("Partner Dataset :", partner_df.shape)

Analyst Dataset : (1407328, 6)
Partner Dataset : (1845559, 6)


## 5. Quick Preview

In [34]:
display(analyst_df.head(2))

display(partner_df.head(2))

,Unnamed: 0,headline,url,publisher,date,stock
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/stocks-that-hit-52-week-highs-on-friday,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/stocks-that-hit-52-week-highs-on-wednesday,Benzinga Insights,2020-06-03 10:45:20-04:00,A


,Unnamed: 0,headline,url,publisher,date,stock
0,2,Agilent Technologies Announces Pricing of $5…… Million of Senior Notes,http://www.gurufocus.com/news/1153187/agilent-technologies-announces-pricing-of-500-million-of-senior-notes,GuruFocus,2020-06-01 00:00:00,A
1,3,Agilent (A) Gears Up for Q2 Earnings: What's in the Cards?,http://www.zacks.com/stock/news/931205/agilent-a-gears-up-for-q2-earnings-whats-in-the-cards?cid=CS-BENZ-FT-analyst_blog|earnings_preview-931205,Zacks,2020-05-18 00:00:00,A


## 6. Add Dataset Source

In [35]:
analyst_df["source"] = "analyst"

partner_df["source"] = "partner"

## 7. Standardize Column Names

In [37]:
rename_dict = {

    "headline": "headline",

    "url": "url",

    "publisher": "publisher",

    "date": "published_date",

    "stock": "ticker"

}

analyst_df.rename(

    columns=rename_dict,

    inplace=True

)

partner_df.rename(

    columns=rename_dict,

    inplace=True

)

## 8. Drop Unnecessary Columns

In [38]:
analyst_df.drop(

    columns=["Unnamed: 0"],

    inplace=True

)

partner_df.drop(

    columns=["Unnamed: 0"],

    inplace=True
)

## 9. Verify Columns

In [39]:
print(analyst_df.columns.tolist())

print(partner_df.columns.tolist())

['headline', 'url', 'publisher', 'published_date', 'ticker', 'source']
['headline', 'url', 'publisher', 'published_date', 'ticker', 'source']


## 10. Convert Date Column

In [40]:
def convert_datetime(series):

    return pd.to_datetime(

        series,

        errors="coerce",

        utc=True,

        format="mixed"

    )

analyst_df["published_date"] = convert_datetime(
    analyst_df["published_date"]
)

partner_df["published_date"] = convert_datetime(
    partner_df["published_date"]
)

In [41]:
print("Analyst Missing Dates :", analyst_df["published_date"].isna().sum())

print("Partner Missing Dates :", partner_df["published_date"].isna().sum())

Analyst Missing Dates : 0
Partner Missing Dates : 0


## 11. Check Date Conversion

In [42]:
print(analyst_df["published_date"].dtype)

print(partner_df["published_date"].dtype)

datetime64[ns, UTC]
datetime64[ns, UTC]


## 12. Merge Datasets

In [43]:
master_df = pd.concat(

    [

        analyst_df,

        partner_df

    ],

    ignore_index=True

)

print(master_df.shape)

(3252887, 6)


## 13. Remove Duplicate News

In [44]:
before = len(master_df)

master_df.drop_duplicates(

    subset=[

        "headline",

        "ticker",

        "published_date"

    ],

    inplace=True

)

after = len(master_df)

print(f"Removed {before-after:,} duplicate news")

Removed 37,571 duplicate news


In [45]:
print(master_df.shape)

print(master_df["published_date"].min())
print(master_df["published_date"].max())

(3215316, 6)
1969-12-31 00:00:00+00:00
2020-06-11 21:12:35+00:00


## 14. Reset Index

In [46]:
master_df.reset_index(

    drop=True,

    inplace=True

)

## 15. Create News ID

In [47]:
master_df.insert(

    0,

    "news_id",

    np.arange(

        1,

        len(master_df)+1

    )

)

## 16. Create Metadata Columns

In [48]:
master_df["year"] = master_df["published_date"].dt.year

master_df["month"] = master_df["published_date"].dt.month

master_df["day"] = master_df["published_date"].dt.day

master_df["weekday"] = master_df["published_date"].dt.day_name()

master_df["hour"] = master_df["published_date"].dt.hour

master_df["headline_length"] = (

    master_df["headline"]

    .str.len()

)

master_df["word_count"] = (

    master_df["headline"]

    .str.split()

    .str.len()

)

## 17. Check Missing Values

In [49]:
master_df.isnull().sum()

,0
news_id,0
headline,0
url,0
publisher,0
published_date,0
ticker,0
source,0
year,0
month,0
day,0


## 18. Final Dataset Overview

In [61]:
print("Rows :", len(master_df))

print("Columns :", len(master_df.columns))

display(master_df.tail())

Rows : 3215315
Columns : 14


,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,weekday,hour,headline_length,word_count
3215310,3215311,Consumer Cyclical Sector Wrap,https://www.benzinga.com/content/12/08/2846030/consumer-cyclical-sector-wrap,webmaster,2012-08-20 00:00:00+00:00,ZX,partner,2012,8,20,Monday,0,29,4
3215311,3215312,Consumer Cyclical Sector Wrap,https://www.benzinga.com/content/12/07/2767124/consumer-cyclical-sector-wrap,webmaster,2012-07-23 00:00:00+00:00,ZX,partner,2012,7,23,Monday,0,29,4
3215312,3215313,Zacks #5 Rank Additions for Monday - Tale of the Tape,http://www.zacks.com/stock/news/73497/here-are-5-stocks-added-to-the-zacks-5-rank-strong-sell-list-for-monday,Zacks,2012-04-23 00:00:00+00:00,ZX,partner,2012,4,23,Monday,0,53,11
3215313,3215314,4 Stock Strategies From Wall Street: Feb. 9 (Update 1),http://www.thestreet.com/story/11409053/1/4-stock-strategies-from-wall-street-feb-9.html?puc=benzinga&cm_ven=BENZINGA,TheStreet.Com,2012-02-09 00:00:00+00:00,ZX,partner,2012,2,9,Thursday,0,54,10
3215314,3215315,4 Stock Strategies From Wall Street: Feb. 9,https://www.benzinga.com/content/thestreet-com/12/02/2330540/4-stock-strategies-from-wall-street-feb-9,webmaster,2012-02-09 00:00:00+00:00,ZX,partner,2012,2,9,Thursday,0,43,8


## 19. Optimize Memory

In [51]:
for col in [

    "ticker",

    "publisher",

    "source",

    "weekday"

]:

    master_df[col] = (

        master_df[col]

        .astype("category")

    )

In [52]:
invalid_dates = master_df[
    master_df["year"] < 2009
]

print(f"Invalid rows found: {len(invalid_dates)}")

display(
    invalid_dates[
        [
            "headline",
            "published_date",
            "stock_symbol" if "stock_symbol" in master_df.columns else "ticker",
            "publisher",
            "source"
        ]
    ]
)

Invalid rows found: 1


,headline,published_date,ticker,publisher,source
2476918,Montpelier Re Holdings Ltd. (MRH): New Analyst Report from Zacks Equity Research - Zacks Equity Research Report,1969-12-31 00:00:00+00:00,MRH,Zacks,partner


In [53]:
print(master_df.shape)

print(master_df["published_date"].min())
print(master_df["published_date"].max())

(3215316, 14)
1969-12-31 00:00:00+00:00
2020-06-11 21:12:35+00:00


In [54]:
master_df = master_df[
    master_df["year"] >= 2009
].copy()

master_df.reset_index(drop=True, inplace=True)

In [55]:
print(master_df.shape)

print(master_df["published_date"].min())
print(master_df["published_date"].max())

(3215315, 14)
2009-02-14 00:00:00+00:00
2020-06-11 21:12:35+00:00


In [56]:
master_df["news_id"] = np.arange(1, len(master_df) + 1)

cols = ["news_id"] + [col for col in master_df.columns if col != "news_id"]

master_df = master_df[cols]

In [62]:
# True = Original dataset contains real publication time
# False = Original dataset contains only the date

master_df["has_time"] = (
    master_df["source"] == "analyst"
)

In [63]:
print(master_df.shape)

print(master_df["published_date"].min())
print(master_df["published_date"].max())

(3215315, 15)
2009-02-14 00:00:00+00:00
2020-06-11 21:12:35+00:00


## 20. Save Master Dataset

In [64]:
save_path = os.path.join(

    INTERIM_DATA,

    "master_news_dataset.parquet"

)

master_df.to_parquet(

    save_path,

    index=False

)

print("Master dataset saved successfully.")

Master dataset saved successfully.


## 21. Summary

In [65]:
summary = pd.DataFrame({

    "Metric":[

        "Total News",

        "Unique Companies",

        "Unique Publishers",

        "Date Range",

        "Sources"

    ],

    "Value":[

        len(master_df),

        master_df["ticker"].nunique(),

        master_df["publisher"].nunique(),

        f"{master_df['published_date'].min()} → {master_df['published_date'].max()}",

        master_df["source"].nunique()

    ]

})

display(summary)

,Metric,Value
0,Total News,3215315
1,Unique Companies,6619
2,Unique Publishers,1047
3,Date Range,2009-02-14 00:00:00+00:00 → 2020-06-11 21:12:35+00:00
4,Sources,2


In [66]:
master_df["published_date"].describe()

,published_date
count,3215315
mean,2016-05-20 21:27:43.025533184+00:00
min,2009-02-14 00:00:00+00:00
25%,2014-06-09 00:00:00+00:00
50%,2016-08-29 00:00:00+00:00
75%,2018-10-27 00:00:00+00:00
max,2020-06-11 21:12:35+00:00


In [67]:
master_df.nsmallest(20, "published_date")

,news_id,headline,url,publisher,published_date,ticker,source,year,month,day,weekday,hour,headline_length,word_count,has_time
872191,872192,How Treasuries and ETFs Work,https://www.benzinga.com/28044/how-treasuries-and-etfs-work,Paco Ahlgren,2009-02-14 00:00:00+00:00,NAV,analyst,2009,2,14,Saturday,0,28,5,True
515502,515503,Update on the Luxury Sector: 2nd Quarter 2009,https://www.benzinga.com/charles-lewis-sizemore-cfa/2009/4/27/update-on-the-luxury-sector-2nd-quarter-2009,Charles Lewis Sizemore CFA,2009-04-27 00:00:00+00:00,FT,analyst,2009,4,27,Monday,0,45,8,True
1378738,1378739,Update on the Luxury Sector: 2nd Quarter 2009,https://www.benzinga.com/charles-lewis-sizemore-cfa/2009/4/27/update-on-the-luxury-sector-2nd-quarter-2009,Charles Lewis Sizemore CFA,2009-04-27 00:00:00+00:00,Y,analyst,2009,4,27,Monday,0,45,8,True
1416,1417,Going Against the Herd,https://www.benzinga.com/charles-lewis-sizemore-cfa/2009/4/29/going-against-the-herd,Charles Lewis Sizemore CFA,2009-04-29 00:00:00+00:00,A,analyst,2009,4,29,Wednesday,0,22,4,True
67112,67113,Charles Sizemore Radio Interview Saturday Morning,https://www.benzinga.com/11218/charles-sizemore-radio-interview-saturday-morning,Charles Lewis Sizemore CFA,2009-05-22 00:00:00+00:00,AM,analyst,2009,5,22,Friday,0,49,6,True
424136,424137,MRM a $15-$20+ stock - FIT new information - JVA 58% super-trades winner,https://www.benzinga.com/superman/2009/5/27/mrm-15-20-stock-fit-new-information-jva-58-super-trades-winner,superman,2009-05-27 00:00:00+00:00,EPS,analyst,2009,5,27,Wednesday,0,72,13,True
424137,424138,"JVA perks to 39% gain, SMCG ready, MRM to continue",https://www.benzinga.com/superman/2009/5/27/jva-perks-39-gain-smcg-ready-mrm-continue,superman,2009-05-27 00:00:00+00:00,EPS,analyst,2009,5,27,Wednesday,0,50,10,True
553959,553960,"JVA perks to 39% gain, SMCG ready, MRM to continue",https://www.benzinga.com/superman/2009/5/27/jva-perks-39-gain-smcg-ready-mrm-continue,superman,2009-05-27 00:00:00+00:00,GMCR,analyst,2009,5,27,Wednesday,0,50,10,True
705505,705506,MRM a $15-$20+ stock - FIT new information - JVA 58% super-trades winner,https://www.benzinga.com/superman/2009/5/27/mrm-15-20-stock-fit-new-information-jva-58-super-trades-winner,superman,2009-05-27 00:00:00+00:00,JVA,analyst,2009,5,27,Wednesday,0,72,13,True
705506,705507,"JVA perks to 39% gain, SMCG ready, MRM to continue",https://www.benzinga.com/superman/2009/5/27/jva-perks-39-gain-smcg-ready-mrm-continue,superman,2009-05-27 00:00:00+00:00,JVA,analyst,2009,5,27,Wednesday,0,50,10,True
